# 🤖 MCP Server + LangGraph Integration Assignment

**Session 13: Model Context Protocol (MCP)**

This notebook demonstrates:
1. Building custom MCP tools with API integration
2. Integrating MCP tools with LangGraph agents
3. Creating an intelligent agent that can use multiple tools

---


In [1]:
import asyncio
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ All imports successful!")
print(f"✅ OpenAI API Key: {'Set' if os.getenv('OPENAI_API_KEY') else 'Not Set'}")


✅ All imports successful!
✅ OpenAI API Key: Set


---

## 🏗️ Activity #1: Building an MCP Server with API Integration

### Currency Converter Tool

I built a **Currency Converter Tool** using the **ExchangeRate-API** (exchangerate-api.com).

**Key Features:**
- Free REST API with no API key required
- Real-time exchange rates for 160+ currencies
- Input validation and error handling
- Beautiful formatted output

**Implementation:** The tool is located in `server.py` (lines 84-162).

### MCP Server Tools Available:

1. **`web_search`** - Search the web using Tavily API
2. **`roll_dice`** - Roll dice with standard notation (e.g., 3d6)
3. **`generate_password`** - Create secure passwords with customizable criteria
4. **`convert_currency`** - Convert between currencies with real-time rates

---


## 🤖 Activity #2: LangGraph Integration with MCP Server

Now let's create a LangGraph agent that can use our MCP tools!

### How it works:
1. Connect to the MCP server via stdio
2. Load all available MCP tools dynamically
3. Create a ReAct agent with ChatOpenAI
4. Agent intelligently uses tools based on user queries


In [2]:
async def create_mcp_agent():
    """
    Creates a LangGraph agent integrated with MCP server tools
    
    Returns:
        tuple: (agent, session, client_context) - The agent and connection objects
    """
    print("🔧 Setting up MCP Server connection...")
    
    # Define MCP server parameters
    server_params = StdioServerParameters(
        command="uv",
        args=[
            "--directory",
            os.path.dirname(os.path.abspath("__file__")),
            "run",
            "server.py"
        ],
    )
    
    # Connect to MCP server
    client_context = stdio_client(server_params)
    read, write = await client_context.__aenter__()
    
    # Create session
    session = ClientSession(read, write)
    await session.__aenter__()
    await session.initialize()
    
    print("✅ Connected to MCP Server!")
    
    # Load MCP tools
    print("\n🔧 Loading MCP tools...")
    tools = await load_mcp_tools(session)
    
    print(f"✅ Loaded {len(tools)} tools:")
    for tool in tools:
        print(f"   - {tool.name}")
    
    # Create LangGraph ReAct agent
    print("\n🤖 Creating LangGraph ReAct Agent...")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_react_agent(llm, tools)
    
    print("✅ Agent created successfully!\n")
    
    return agent, session, client_context

print("✅ Function defined: create_mcp_agent()")


✅ Function defined: create_mcp_agent()


### Testing the Agent

Let's create a helper function to test our agent with various queries.


In [3]:
async def test_agent_query(agent, query):
    """
    Test the agent with a single query
    
    Args:
        agent: The LangGraph agent
        query: User query string
    """
    print(f"\n{'='*70}")
    print(f"📝 Query: {query}")
    print("="*70)
    
    try:
        response = await agent.ainvoke({"messages": [("user", query)]})
        final_message = response["messages"][-1]
        print(f"\n🤖 Response: {final_message.content}\n")
        return final_message.content
    except Exception as e:
        print(f"\n❌ Error: {e}\n")
        return None

print("✅ Function defined: test_agent_query()")


✅ Function defined: test_agent_query()


---

## 🧪 Test 1: Currency Conversion (USD to JPY)


In [ ]:
# Create agent and run test
agent, session, client_context = await create_mcp_agent()

# Test: Currency Conversion
result = await test_agent_query(agent, "Convert 100 USD to JPY")


: 

---

## 🧪 Test 2: Password Generation


In [5]:
# Test: Password Generation
result = await test_agent_query(
    agent, 
    "Generate a 12-character password with only letters and numbers"
)



📝 Query: Generate a 12-character password with only letters and numbers

🤖 Response: Here is your generated password: **1fiDXZj32m0L** (12 characters, containing only letters and numbers).



---

## 🧪 Test 3: Dice Rolling


In [6]:
# Test: Dice Rolling
result = await test_agent_query(agent, "Roll 3d6 dice")



📝 Query: Roll 3d6 dice

🤖 Response: You rolled three six-sided dice (3d6) and got the results: 6, 5, and 3, for a total of 14.



---

## 🧪 Test 4: Complex Currency Query


In [7]:
# Test: Complex Currency Conversion
result = await test_agent_query(
    agent,
    "What's the exchange rate from EUR to GBP for 500 euros?"
)



📝 Query: What's the exchange rate from EUR to GBP for 500 euros?

🤖 Response: The exchange rate from EUR to GBP for 500 euros is as follows:

- **Converted Amount:** 435.47 GBP
- **Exchange Rate:** 1 EUR = 0.870942 GBP

This information was last updated on October 24, 2025.

